# Top-K Guide Evaluation

Visualizes `top_k.json` output from `guide-eval`: how do the top-k guides (ranked by ZS distance, structural distance, or randomly selected) perform when combined? Also compares pure random selection vs random with a helper guide (best ZS or best structural).

Run with: `uv run jupyter notebook top_k_eval.ipynb`

In [ ]:
import json

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from helpers import find_latest_run, best_single_from_trial, parse_random_trials

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 8)})

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
run_dir = find_latest_run("enumerate")
json_file = run_dir / "top_k.json"

print(f"Using: {json_file}")

with open(json_file) as f:
    raw = json.load(f)

print(f"{len(raw)} goal(s) in dataset")

In [ ]:
# ── Parse into flat DataFrames ─────────────────────────────────────────
# Deterministic strategies: zs, structural
DETERMINISTIC_STRATEGIES = ["zs", "structural"]
# Random variants: pure random, random + best ZS helper, random + best structural helper
RANDOM_STRATEGIES = ["random", "random_with_best_zs", "random_with_best_structural"]

det_rows = []

for entry in raw:
    goal = entry["goal"]
    for strategy in DETERMINISTIC_STRATEGIES:
        for item in entry.get(strategy, []):
            det_rows.append(
                {
                    "goal": goal,
                    "strategy": strategy,
                    "k": item["k"],
                    "best_single_iters": item["best_single_iters"],
                    "combined_iters": item["combined_iters"],
                    "combined_nodes": item["combined_nodes"],
                }
            )

rand_rows = []
for rand_strategy in RANDOM_STRATEGIES:
    rand_rows.extend(parse_random_trials(raw, rand_strategy, rand_strategy))

df_det = pl.DataFrame(det_rows)
df_rand = pl.DataFrame(rand_rows)

# Short goal labels for plotting
goal_labels = {
    g: g if len(g) <= 40 else g[:37] + "..." for g in df_det["goal"].unique().to_list()
}

# Filter out goals with no data (all strategies empty)
active_goals = (
    df_det.filter(pl.col("combined_iters").is_not_null())["goal"].unique().to_list()
)
df_det = df_det.filter(pl.col("goal").is_in(active_goals))
df_rand = df_rand.filter(pl.col("goal").is_in(active_goals))

print(f"{len(active_goals)} goal(s) with reachable guides")
print(f"Deterministic rows: {len(df_det)}, Random trial rows: {len(df_rand)}")
print(f"Random strategies: {df_rand['strategy'].unique().to_list()}")
df_det.head(10)

## Combined iterations vs k (per strategy)

For each strategy, how does combining the top-k guides affect the number of iterations needed to reach the goal?

In [ ]:
# ── Line plot: combined_iters vs k, one line per strategy, averaged over goals ──
k_values = sorted(df_det["k"].unique().to_list())

DET_COLORS = {
    "zs": "steelblue",
    "structural": "coral",
}
DET_LABELS = {
    "zs": "ZS distance",
    "structural": "structural distance",
}

RAND_COLORS = {
    "random": "gray",
    "random_with_best_zs": "mediumpurple",
    "random_with_best_structural": "darkorange",
}
RAND_LABELS = {
    "random": "random",
    "random_with_best_zs": "random + best ZS",
    "random_with_best_structural": "random + best structural",
}

STRATEGY_COLORS = {**DET_COLORS, **RAND_COLORS}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric, ax in zip(["combined_iters", "combined_nodes"], axes):
    for strategy in DETERMINISTIC_STRATEGIES:
        sub = df_det.filter(pl.col("strategy") == strategy)
        agg = (
            sub.group_by("k")
            .agg(pl.col(metric).mean().alias("mean"), pl.col(metric).std().alias("std"))
            .sort("k")
        )
        ks = agg["k"].to_numpy()
        means = agg["mean"].to_numpy()
        stds = agg["std"].fill_null(0).to_numpy()
        color = DET_COLORS[strategy]
        ax.plot(ks, means, marker="o", color=color, label=DET_LABELS[strategy])
        ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=color)

    # Random variants
    for rand_strat in RANDOM_STRATEGIES:
        rsub = df_rand.filter(pl.col("strategy") == rand_strat)
        rand_agg = (
            rsub.group_by("k")
            .agg(pl.col(metric).mean().alias("mean"), pl.col(metric).std().alias("std"))
            .sort("k")
        )
        if len(rand_agg) == 0:
            continue
        ks = rand_agg["k"].to_numpy()
        means = rand_agg["mean"].to_numpy()
        stds = rand_agg["std"].fill_null(0).to_numpy()
        color = RAND_COLORS[rand_strat]
        ax.plot(
            ks, means, marker="s", ls="--", color=color, label=RAND_LABELS[rand_strat]
        )
        ax.fill_between(ks, means - stds, means + stds, alpha=0.1, color=color)

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs k (mean ± std over goals)")
    ax.legend(fontsize=8)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)

fig.tight_layout()
plt.show()

## Per-goal: combined iterations vs k

In [ ]:
# ── Per-goal line plots ────────────────────────────────────────────────
n_goals = len(active_goals)
cols = min(3, n_goals)
rows = (n_goals + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows), squeeze=False)

for idx, goal in enumerate(sorted(active_goals)):
    ax = axes[idx // cols][idx % cols]
    for strategy in DETERMINISTIC_STRATEGIES:
        sub = df_det.filter(
            (pl.col("goal") == goal) & (pl.col("strategy") == strategy)
        ).sort("k")
        if len(sub) == 0:
            continue
        ax.plot(
            sub["k"].to_numpy(),
            sub["combined_iters"].to_numpy(),
            marker="o",
            color=DET_COLORS[strategy],
            label=DET_LABELS[strategy],
        )

    # Random variants
    for rand_strat in RANDOM_STRATEGIES:
        rsub = df_rand.filter(
            (pl.col("goal") == goal) & (pl.col("strategy") == rand_strat)
        )
        if len(rsub) == 0:
            continue
        ragg = (
            rsub.group_by("k")
            .agg(
                pl.col("combined_iters").mean().alias("mean"),
                pl.col("combined_iters").std().alias("std"),
            )
            .sort("k")
        )
        ks = ragg["k"].to_numpy()
        means = ragg["mean"].to_numpy()
        stds = ragg["std"].fill_null(0).to_numpy()
        color = RAND_COLORS[rand_strat]
        ax.plot(
            ks, means, marker="s", ls="--", color=color, label=RAND_LABELS[rand_strat]
        )
        ax.fill_between(ks, means - stds, means + stds, alpha=0.1, color=color)

    ax.set_xlabel("k")
    ax.set_ylabel("combined_iters")
    ax.set_title(goal_labels[goal], fontsize=9)
    ax.legend(fontsize=6)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)

for j in range(n_goals, rows * cols):
    axes[j // cols][j % cols].set_visible(False)

fig.suptitle("Combined iterations vs k (per goal)", fontsize=13)
fig.tight_layout()
plt.show()

## Box plots: combined iterations by strategy at each k

Distribution across goals of how many iterations it takes when combining top-k guides.

In [ ]:
# ── Box plots of combined_iters by strategy, grouped by k ─────────────
all_strategies = DETERMINISTIC_STRATEGIES + RANDOM_STRATEGIES
ALL_LABELS = {**DET_LABELS, **RAND_LABELS}
n_k = len(k_values)
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for strategy in DETERMINISTIC_STRATEGIES:
        vals = (
            df_det.filter((pl.col("k") == k) & (pl.col("strategy") == strategy))[
                "combined_iters"
            ]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(DET_LABELS[strategy])
            colors.append(DET_COLORS[strategy])
    for rand_strat in RANDOM_STRATEGIES:
        vals = (
            df_rand.filter((pl.col("k") == k) & (pl.col("strategy") == rand_strat))[
                "combined_iters"
            ]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(RAND_LABELS[rand_strat])
            colors.append(RAND_COLORS[rand_strat])

    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("combined_iters" if ki == 0 else "")
    ax.tick_params(axis="x", rotation=45, labelsize=7)

fig.suptitle("Combined iterations by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Box plots: combined nodes by strategy at each k

In [ ]:
# ── Box plots of combined_nodes by strategy, grouped by k ─────────────
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for strategy in DETERMINISTIC_STRATEGIES:
        vals = (
            df_det.filter((pl.col("k") == k) & (pl.col("strategy") == strategy))[
                "combined_nodes"
            ]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(DET_LABELS[strategy])
            colors.append(DET_COLORS[strategy])
    for rand_strat in RANDOM_STRATEGIES:
        vals = (
            df_rand.filter((pl.col("k") == k) & (pl.col("strategy") == rand_strat))[
                "combined_nodes"
            ]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(RAND_LABELS[rand_strat])
            colors.append(RAND_COLORS[rand_strat])

    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("combined_nodes" if ki == 0 else "")
    ax.tick_params(axis="x", rotation=45, labelsize=7)

fig.suptitle("Combined nodes by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Best single guide vs combined (top-k)

Does combining guides actually help vs just using the best single guide in the set?

In [ ]:
# ── Scatter: best_single_iters vs combined_iters ───────────────────────
fig, axes = plt.subplots(
    1,
    len(DETERMINISTIC_STRATEGIES),
    figsize=(6 * len(DETERMINISTIC_STRATEGIES), 5),
    squeeze=False,
)

for si, strategy in enumerate(DETERMINISTIC_STRATEGIES):
    ax = axes[0][si]
    sub = df_det.filter(pl.col("strategy") == strategy).drop_nulls(
        subset=["best_single_iters", "combined_iters"]
    )
    if len(sub) == 0:
        ax.set_visible(False)
        continue
    x = sub["best_single_iters"].to_numpy()
    y = sub["combined_iters"].to_numpy()
    ks = sub["k"].to_numpy()
    scatter = ax.scatter(
        x,
        y,
        c=np.log2(ks),
        cmap="viridis",
        s=40,
        alpha=0.7,
        edgecolors="k",
        linewidths=0.5,
    )
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("log2(k)")

    # Diagonal reference
    lims = [min(x.min(), y.min()) - 0.5, max(x.max(), y.max()) + 0.5]
    ax.plot(lims, lims, "k--", alpha=0.3, lw=1)
    ax.set_xlabel("best_single_iters")
    ax.set_ylabel("combined_iters")
    ax.set_title(f"{DET_LABELS[strategy]}: single vs combined")
    ax.set_aspect("equal")

fig.suptitle("Best single guide iters vs combined top-k iters", fontsize=13)
fig.tight_layout()
plt.show()

## Improvement from combining: (best_single - combined) vs k

In [ ]:
# ── Improvement (best_single_iters - combined_iters) vs k ──────────────
fig, ax = plt.subplots(figsize=(10, 5))

for strategy in DETERMINISTIC_STRATEGIES:
    sub = (
        df_det.filter(pl.col("strategy") == strategy)
        .drop_nulls(subset=["best_single_iters", "combined_iters"])
        .with_columns(
            (pl.col("best_single_iters") - pl.col("combined_iters")).alias(
                "improvement"
            )
        )
    )
    agg = (
        sub.group_by("k")
        .agg(
            pl.col("improvement").mean().alias("mean"),
            pl.col("improvement").std().alias("std"),
        )
        .sort("k")
    )
    ks = agg["k"].to_numpy()
    means = agg["mean"].to_numpy()
    stds = agg["std"].fill_null(0).to_numpy()
    color = DET_COLORS[strategy]
    ax.plot(ks, means, marker="o", color=color, label=DET_LABELS[strategy])
    ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=color)

# Random variants
for rand_strat in RANDOM_STRATEGIES:
    rsub = (
        df_rand.filter(pl.col("strategy") == rand_strat)
        .drop_nulls(subset=["best_single_iters", "combined_iters"])
        .with_columns(
            (pl.col("best_single_iters") - pl.col("combined_iters")).alias(
                "improvement"
            )
        )
    )
    ragg = (
        rsub.group_by("k")
        .agg(
            pl.col("improvement").mean().alias("mean"),
            pl.col("improvement").std().alias("std"),
        )
        .sort("k")
    )
    if len(ragg) == 0:
        continue
    ks = ragg["k"].to_numpy()
    means = ragg["mean"].to_numpy()
    stds = ragg["std"].fill_null(0).to_numpy()
    color = RAND_COLORS[rand_strat]
    ax.plot(ks, means, marker="s", ls="--", color=color, label=RAND_LABELS[rand_strat])
    ax.fill_between(ks, means - stds, means + stds, alpha=0.1, color=color)

ax.axhline(0, color="black", ls=":", alpha=0.3)
ax.set_xlabel("k")
ax.set_ylabel("improvement (best_single - combined iters)")
ax.set_title("Iteration improvement from combining top-k guides")
ax.legend(fontsize=8)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
fig.tight_layout()
plt.show()

## Nodes growth: how does egraph size scale with k?

In [ ]:
# ── combined_nodes vs k ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for strategy in DETERMINISTIC_STRATEGIES:
    sub = df_det.filter(pl.col("strategy") == strategy).drop_nulls(
        subset=["combined_nodes"]
    )
    agg = (
        sub.group_by("k")
        .agg(
            pl.col("combined_nodes").mean().alias("mean"),
            pl.col("combined_nodes").std().alias("std"),
        )
        .sort("k")
    )
    ks = agg["k"].to_numpy()
    means = agg["mean"].to_numpy()
    stds = agg["std"].fill_null(0).to_numpy()
    color = DET_COLORS[strategy]
    ax.plot(ks, means, marker="o", color=color, label=DET_LABELS[strategy])
    ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=color)

# Random variants
for rand_strat in RANDOM_STRATEGIES:
    ragg = (
        df_rand.filter(pl.col("strategy") == rand_strat)
        .drop_nulls(subset=["combined_nodes"])
        .group_by("k")
        .agg(
            pl.col("combined_nodes").mean().alias("mean"),
            pl.col("combined_nodes").std().alias("std"),
        )
        .sort("k")
    )
    if len(ragg) == 0:
        continue
    ks = ragg["k"].to_numpy()
    means = ragg["mean"].to_numpy()
    stds = ragg["std"].fill_null(0).to_numpy()
    color = RAND_COLORS[rand_strat]
    ax.plot(ks, means, marker="s", ls="--", color=color, label=RAND_LABELS[rand_strat])
    ax.fill_between(ks, means - stds, means + stds, alpha=0.1, color=color)

ax.set_xlabel("k")
ax.set_ylabel("combined_nodes")
ax.set_title("Egraph nodes vs k (mean ± std over goals)")
ax.legend(fontsize=8)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
fig.tight_layout()
plt.show()

## Random baseline: pure random vs random with helper

Box plots comparing trial outcomes: does adding a single "best" guide (by ZS or structural distance) to a random set help?

In [ ]:
# ── Random variant comparison box plots ────────────────────────────────
rand_k_values = sorted(df_rand["k"].unique().to_list())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for metric, ax in zip(["combined_iters", "combined_nodes"], axes):
    positions = []
    box_data = []
    colors_list = []
    tick_positions = []
    tick_labels = []
    n_strats = len(RANDOM_STRATEGIES)
    width = 0.8 / n_strats

    for ki, k in enumerate(rand_k_values):
        center = ki * 1.5
        tick_positions.append(center)
        tick_labels.append(str(k))
        for si, rand_strat in enumerate(RANDOM_STRATEGIES):
            vals = (
                df_rand.filter((pl.col("k") == k) & (pl.col("strategy") == rand_strat))[
                    metric
                ]
                .drop_nulls()
                .to_numpy()
            )
            if len(vals) > 0:
                pos = center + (si - n_strats / 2 + 0.5) * width
                positions.append(pos)
                box_data.append(vals)
                colors_list.append(RAND_COLORS[rand_strat])

    if box_data:
        bp = ax.boxplot(
            box_data, positions=positions, widths=width * 0.8, patch_artist=True
        )
        for patch, color in zip(bp["boxes"], colors_list):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    ax.set_xlabel("k")
    ax.set_ylabel(metric)
    ax.set_title(f"Random variants: {metric} across trials")

    # Legend
    from matplotlib.patches import Patch

    legend_patches = [
        Patch(facecolor=RAND_COLORS[s], alpha=0.6, label=RAND_LABELS[s])
        for s in RANDOM_STRATEGIES
    ]
    ax.legend(handles=legend_patches, fontsize=8)

fig.tight_layout()
plt.show()

## Strategy comparison heatmap

For each goal x k, which strategy achieves the fewest combined iterations?

In [ ]:
# ── Heatmap: combined_iters per goal x strategy at selected k ──────────
# Aggregate random variants into means
rand_means = (
    df_rand.group_by("goal", "k", "strategy")
    .agg(pl.col("combined_iters").mean().alias("combined_iters"))
    .with_columns(pl.col("strategy").replace(RAND_LABELS).alias("strategy"))
)
combined = pl.concat(
    [
        df_det.select(
            "goal",
            "k",
            pl.col("strategy").replace(DET_LABELS),
            pl.col("combined_iters").cast(pl.Float64),
        ),
        rand_means.select("goal", "k", "strategy", "combined_iters"),
    ]
)

for k in [5, 10, 50, 100]:
    sub = combined.filter(pl.col("k") == k)
    if len(sub) == 0:
        continue
    pivot = sub.pivot(on="strategy", index="goal", values="combined_iters").sort("goal")
    strat_cols = [c for c in pivot.columns if c != "goal"]
    if not strat_cols:
        continue

    goal_names = [goal_labels.get(g, g) for g in pivot["goal"].to_list()]
    mat = pivot.select(strat_cols).to_numpy().astype(float)

    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(goal_names))))
    im = ax.imshow(mat, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(strat_cols)))
    ax.set_xticklabels(strat_cols, rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(len(goal_names)))
    ax.set_yticklabels(goal_names, fontsize=7)

    for i in range(len(goal_names)):
        for j in range(len(strat_cols)):
            val = mat[i, j]
            if not np.isnan(val):
                ax.text(
                    j,
                    i,
                    f"{val:.0f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if val > np.nanmedian(mat) else "black",
                )

    fig.colorbar(im, ax=ax, label="combined_iters")
    ax.set_title(f"Combined iterations by strategy (k={k})")
    fig.tight_layout()
    plt.show()

## Summary table

In [ ]:
# ── Summary: best strategy per k ──────────────────────────────────────
rand_mean_full = (
    df_rand.group_by("goal", "k", "strategy")
    .agg(
        pl.col("combined_iters").mean().alias("combined_iters"),
        pl.col("combined_nodes").mean().alias("combined_nodes"),
        pl.col("best_single_iters").mean().alias("best_single_iters"),
    )
    .with_columns(pl.col("strategy").replace(RAND_LABELS).alias("strategy"))
)
all_data = pl.concat(
    [
        df_det.select(
            "goal",
            "k",
            pl.col("strategy").replace(DET_LABELS),
            pl.col("combined_iters").cast(pl.Float64),
            pl.col("combined_nodes").cast(pl.Float64),
            pl.col("best_single_iters").cast(pl.Float64),
        ),
        rand_mean_full.select(
            "goal",
            "k",
            "strategy",
            "combined_iters",
            "combined_nodes",
            "best_single_iters",
        ),
    ]
)

summary = (
    all_data.group_by("strategy", "k")
    .agg(
        pl.col("combined_iters").mean().round(1).alias("avg_combined_iters"),
        pl.col("combined_nodes").mean().round(0).alias("avg_combined_nodes"),
        pl.col("best_single_iters").mean().round(1).alias("avg_best_single"),
    )
    .sort("k", "strategy")
)
summary

## Helper guide effect

How much does adding a single "best" guide (by ZS or structural distance) improve over pure random selection? Negative values mean the helper reduced iterations.

In [ ]:
# ── Helper effect: delta between pure random and random+helper ─────────
# Compute per-goal, per-k means for each random strategy
rand_per_goal_k = df_rand.group_by("goal", "k", "strategy").agg(
    pl.col("combined_iters").mean().alias("mean_iters"),
    pl.col("combined_nodes").mean().alias("mean_nodes"),
)

pure_random = rand_per_goal_k.filter(pl.col("strategy") == "random").select(
    "goal",
    "k",
    pl.col("mean_iters").alias("random_iters"),
    pl.col("mean_nodes").alias("random_nodes"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for helper_strat, color, label in [
    ("random_with_best_zs", "mediumpurple", "random + best ZS"),
    ("random_with_best_structural", "darkorange", "random + best structural"),
]:
    helper = rand_per_goal_k.filter(pl.col("strategy") == helper_strat).select(
        "goal",
        "k",
        pl.col("mean_iters").alias("helper_iters"),
        pl.col("mean_nodes").alias("helper_nodes"),
    )
    delta = pure_random.join(helper, on=["goal", "k"]).with_columns(
        (pl.col("random_iters") - pl.col("helper_iters")).alias("delta_iters"),
        (pl.col("random_nodes") - pl.col("helper_nodes")).alias("delta_nodes"),
    )

    for metric, ax in zip(["delta_iters", "delta_nodes"], axes):
        agg = (
            delta.group_by("k")
            .agg(
                pl.col(metric).mean().alias("mean"),
                pl.col(metric).std().alias("std"),
            )
            .sort("k")
        )
        ks = agg["k"].to_numpy()
        means = agg["mean"].to_numpy()
        stds = agg["std"].fill_null(0).to_numpy()
        ax.plot(ks, means, marker="o", color=color, label=label)
        ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=color)

for ax, ylabel in zip(axes, ["iters saved", "nodes saved"]):
    ax.axhline(0, color="black", ls=":", alpha=0.3)
    ax.set_xlabel("k (number of random guides)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Helper effect: {ylabel} (pure random - random+helper)")
    ax.legend(fontsize=8)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)

fig.tight_layout()
plt.show()

In [ ]:
# ── Reachability rate: fraction of trials that reached the goal ────────
fig, ax = plt.subplots(figsize=(10, 5))

for rand_strat in RANDOM_STRATEGIES:
    rsub = df_rand.filter(pl.col("strategy") == rand_strat)
    reach = (
        rsub.group_by("k")
        .agg(
            pl.col("combined_iters").is_not_null().mean().alias("reach_rate"),
        )
        .sort("k")
    )
    if len(reach) == 0:
        continue
    color = RAND_COLORS[rand_strat]
    ax.plot(
        reach["k"].to_numpy(),
        reach["reach_rate"].to_numpy() * 100,
        marker="o",
        color=color,
        label=RAND_LABELS[rand_strat],
    )

# Deterministic strategies
for strategy in DETERMINISTIC_STRATEGIES:
    sub = df_det.filter(pl.col("strategy") == strategy)
    reach = (
        sub.group_by("k")
        .agg(pl.col("combined_iters").is_not_null().mean().alias("reach_rate"))
        .sort("k")
    )
    color = DET_COLORS[strategy]
    ax.plot(
        reach["k"].to_numpy(),
        reach["reach_rate"].to_numpy() * 100,
        marker="s",
        ls="-",
        color=color,
        label=DET_LABELS[strategy],
    )

ax.set_xlabel("k (number of guides)")
ax.set_ylabel("reachability (%)")
ax.set_title("Fraction of goals/trials reached vs k")
ax.legend(fontsize=8)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
ax.set_ylim(-5, 105)
fig.tight_layout()
plt.show()